In [ ]:
%load_ext rpy2.ipython

In [ ]:
import pandas as pd

import src
from src.load import DataLoader

r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [ ]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

In [ ]:
# Load all the Data

dl = DataLoader()

videos = dl.channels().join(dl.videos(), "channel_id").to_pandas()
sents = dl.sentences().join(dl.popbert(), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# Dataset Summary Table

In [ ]:
channel_overview = (
    videos.merge(sents, on="video_id")
    .groupby("channel", observed=True)
    .agg(
        ch_videos=("channel", "size"),
        ch_followers=("channel_followers", "first"),
        n_sentences=("n_sents", "sum"),
        n_elite=("n_elite", "sum"),
        n_pplcentr=("n_pplcentr", "sum"),
        avg_likes=("video_likes", "mean"),
        avg_views=("video_views", "mean"),
        avg_duration=("video_duration", "mean"),
        avg_comments=("video_comments", "mean"),
        first_video=("video_uploadtime", "min"),
        latest_video=("video_uploadtime", "max"),
    )
)

In [ ]:
channel_overview

In [ ]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

In [ ]:
# number of videos

count_videos = videos.video_id.size
print(f"Total number of valid videos: {count_videos}")

In [ ]:
# number of sentencs

count_sents = channel_overview.n_sentences.sum()
print(f"Total number of valid sentences: {count_sents}")

In [ ]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T
summary_table

In [ ]:
path = src.OUT / "tables/dataset_summary.csv"
summary_table.to_csv(path)

# View Count Violin Plot

In [ ]:
df = videos.merge(sents, on="video_id")

In [ ]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=2)

ggsave(here(r_out, "/figures/view_count_violin.svg"))

In [ ]:
%%R -i df -w 800 -h 1000

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=1)

ggsave(here(r_out, "/figures/view_count_violin_vertical.svg"))
